# Explanation Pairwise Test

This notebook contains the pairwise comparison code used to evaluate explanation quality across detection/explanation settings.

The code cell below preserves the original Python script logic without modifying the implementation.


In [ ]:
import argparse
import csv
import json
import os
import random
import re
import shutil
import sys
import time
import uuid
from pathlib import Path

import requests


REPO_ROOT = Path(__file__).resolve().parents[1]
BACKEND_ROOT = REPO_ROOT / "backend" / "deepfake"


def load_dotenv(path):
    if not path.exists():
        return

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


load_dotenv(REPO_ROOT / ".env")

sys.path.insert(0, str(BACKEND_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deepfake.settings")

import django  # noqa: E402

django.setup()

from django.conf import settings  # noqa: E402

from detection.services.explainability import run_gradcam  # noqa: E402
from detection.services.inference import run_inference  # noqa: E402
from detection.services.preprocessing import run_preprocessing  # noqa: E402


SETTING_ORDER = [
    "binary_only",
    "binary_method",
    "binary_method_roi",
]

SETTING_LABELS = {
    "binary_only": "Binary only",
    "binary_method": "Binary + Method",
    "binary_method_roi": "Binary + Method + ROI",
}

PAIRWISE_COMPARISONS = [
    ("binary_only", "binary_method"),
    ("binary_method", "binary_method_roi"),
    ("binary_only", "binary_method_roi"),
]


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def collect_videos(data_root):
    extensions = {".mp4", ".avi", ".mov", ".mkv"}
    return sorted(
        path
        for path in Path(data_root).rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


def infer_label_from_path(path):
    parts = {part.lower() for part in path.parts}
    if "fake" in parts:
        return "FAKE"
    if "real" in parts:
        return "REAL"
    return "UNKNOWN"


def sample_videos(data_root, sample_size, seed, balanced=False):
    videos = collect_videos(data_root)
    if not videos:
        raise ValueError(f"No video files found under: {data_root}")

    rng = random.Random(seed)

    if balanced:
        real_videos = [path for path in videos if infer_label_from_path(path) == "REAL"]
        fake_videos = [path for path in videos if infer_label_from_path(path) == "FAKE"]
        rng.shuffle(real_videos)
        rng.shuffle(fake_videos)

        per_class = sample_size // 2
        selected = real_videos[:per_class] + fake_videos[:per_class]

        if len(selected) < sample_size:
            selected_paths = set(selected)
            leftovers = [path for path in videos if path not in selected_paths]
            rng.shuffle(leftovers)
            selected.extend(leftovers[:sample_size - len(selected)])

        rng.shuffle(selected)
        return selected

    rng.shuffle(videos)
    return videos[: min(sample_size, len(videos))]


def iter_candidate_videos(data_root, seed, balanced=False):
    videos = collect_videos(data_root)
    if not videos:
        raise ValueError(f"No video files found under: {data_root}")

    rng = random.Random(seed)

    if not balanced:
        rng.shuffle(videos)
        return videos

    real_videos = [path for path in videos if infer_label_from_path(path) == "REAL"]
    fake_videos = [path for path in videos if infer_label_from_path(path) == "FAKE"]
    other_videos = [path for path in videos if infer_label_from_path(path) == "UNKNOWN"]

    rng.shuffle(real_videos)
    rng.shuffle(fake_videos)
    rng.shuffle(other_videos)

    interleaved = []
    max_len = max(len(real_videos), len(fake_videos))
    for index in range(max_len):
        if index < len(real_videos):
            interleaved.append(real_videos[index])
        if index < len(fake_videos):
            interleaved.append(fake_videos[index])

    interleaved.extend(other_videos)
    return interleaved


def copy_video_to_media(video_path):
    media_root = Path(settings.MEDIA_ROOT)
    media_root.mkdir(parents=True, exist_ok=True)

    filename = f"{uuid.uuid4().hex}_{video_path.name}"
    save_path = media_root / filename
    shutil.copy2(video_path, save_path)
    return filename, str(save_path)


def compact_roi_summary(roi_result):
    regions = [region for region in roi_result.get("facial_region", []) if region != "None"]
    detection_probability = roi_result.get("detection_probability", {})
    first_detection_rate = roi_result.get("first_detection_rate", {})
    first_detection_count = roi_result.get("first_detection_count", {})
    second_detection_count = roi_result.get("second_detection_count", {})

    top_regions = sorted(
        [
            (
                region,
                detection_probability.get(region, 0.0),
                first_detection_rate.get(region, 0.0),
                first_detection_count.get(region, 0),
                second_detection_count.get(region, 0),
            )
            for region in regions
        ],
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        "top_roi_by_contribution": [
            {
                "region": region,
                "contribution_percent": contribution,
                "first_detection_rate_percent": first_rate,
                "first_count": first_count,
                "second_count": second_count,
            }
            for region, contribution, first_rate, first_count, second_count in top_regions[:3]
        ],
        "none_first_count": first_detection_count.get("None", 0),
        "none_second_count": second_detection_count.get("None", 0),
    }


def build_evidence(video_path, result, roi_result, setting):
    evidence = {
        "video_name": video_path.name,
        "binary_prediction": result["Prediction"],
        "probability_percent": result["Probability"],
    }

    if setting in {"binary_method", "binary_method_roi"}:
        evidence["method_prediction"] = result["Method"]

    if setting == "binary_method_roi":
        evidence["roi_statistics"] = compact_roi_summary(roi_result)

    return evidence


def build_explanation_prompt(evidence, setting):
    if setting == "binary_only":
        available = "binary real/fake prediction and confidence only"
    elif setting == "binary_method":
        available = "binary prediction, confidence, and manipulation-method prediction"
    else:
        available = "binary prediction, confidence, manipulation-method prediction, and ROI statistics"

    return (
        "You are explaining a deepfake detection result to a non-expert user.\n"
        f"Available evidence: {available}.\n"
        "Use only the provided evidence. Do not invent visual details or facial regions that are not present.\n"
        "Write a concise explanation in 3 numbered points.\n\n"
        f"Evidence JSON:\n{json.dumps(evidence, ensure_ascii=False, indent=2)}"
    )


def ollama_generate(prompt, url, model, timeout, temperature=0.0, num_predict=500):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
        },
    }

    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    return data.get("response", "").strip()


def openai_generate(prompt, model, timeout, temperature=0.0, num_predict=500, max_retries=5):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set. Put it in .env or export it before running.")

    payload = {
        "model": model,
        "input": prompt,
        "temperature": temperature,
        "max_output_tokens": num_predict,
    }
    response = None
    for attempt in range(max_retries + 1):
        response = requests.post(
            "https://api.openai.com/v1/responses",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=timeout,
        )
        if response.status_code != 429:
            break

        retry_after = response.headers.get("retry-after")
        if retry_after:
            sleep_sec = float(retry_after)
        else:
            sleep_sec = min(60, 2 ** attempt)
        log(f"OpenAI rate limit hit. Retrying in {sleep_sec:.1f}s ({attempt + 1}/{max_retries})")
        time.sleep(sleep_sec)

    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        message = response.text[:1000] if response is not None else str(exc)
        raise RuntimeError(f"OpenAI API request failed: {message}") from exc
    data = response.json()

    if data.get("output_text"):
        return data["output_text"].strip()

    parts = []
    for item in data.get("output", []):
        for content in item.get("content", []):
            if content.get("type") in {"output_text", "text"}:
                parts.append(content.get("text", ""))
    return "".join(parts).strip()


def generate_text(prompt, provider, url, model, timeout, temperature=0.0, num_predict=500):
    if provider == "openai":
        return openai_generate(
            prompt,
            model=model,
            timeout=timeout,
            temperature=temperature,
            num_predict=num_predict,
        )

    return ollama_generate(
        prompt,
        url=url,
        model=model,
        timeout=timeout,
        temperature=temperature,
        num_predict=num_predict,
    )


def parse_json_object(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))

def build_pairwise_prompt(video_name, prediction, setting_a, explanation_a, setting_b, explanation_b):
    return (
        "You are evaluating explanations for a deepfake detection system.\n"
        "Two explanations are generated for the same video and the same model decision.\n"
        "Choose which explanation better helps a non-expert user understand why the model predicted REAL or FAKE.\n\n"
        "Evaluate the explanations using these neutral criteria:\n"
        "1. Decision justification: whether the explanation clearly supports the model's REAL/FAKE prediction.\n"
        "2. Faithfulness: whether the explanation relies only on the provided evidence and avoids unsupported claims.\n"
        "3. Specificity: whether the explanation provides concrete evidence rather than generic statements.\n"
        "4. Completeness: whether the explanation uses the available evidence sufficiently without overclaiming.\n"
        "5. Interpretability: whether the explanation is understandable to a non-expert user.\n\n"
        "Important instructions:\n"
        "- Do not prefer an explanation simply because it is longer or contains more categories of evidence.\n"
        "- Do not penalize an explanation simply because less evidence was available for that setting.\n"
        "- Prefer the explanation with better justified, more faithful, and more understandable reasoning.\n"
        "- If both explanations are equally good under these criteria, choose \"tie\".\n\n"
        "Return only valid JSON:\n"
        "{\n"
        '  "winner": "A",\n'
        '  "confidence": 1,\n'
        '  "brief_reason": "one sentence"\n'
        "}\n"
        'The winner must be one of "A", "B", or "tie". Confidence is 1 to 5.\n\n'
        f"Video: {video_name}\n"
        f"Model prediction: {prediction}\n\n"
        f"Explanation A setting: {SETTING_LABELS[setting_a]}\n"
        f"Explanation A:\n{explanation_a}\n\n"
        f"Explanation B setting: {SETTING_LABELS[setting_b]}\n"
        f"Explanation B:\n{explanation_b}"
    )


def judge_pairwise(video_name, prediction, setting_a, explanation_a, setting_b, explanation_b, provider, url, model, timeout):
    prompt = build_pairwise_prompt(
        video_name,
        prediction,
        setting_a,
        explanation_a,
        setting_b,
        explanation_b,
    )
    raw = generate_text(
        prompt,
        provider=provider,
        url=url,
        model=model,
        timeout=timeout,
        temperature=0.0,
        num_predict=200,
    )
    result = parse_json_object(raw)
    winner = str(result.get("winner", "")).strip()
    if winner not in {"A", "B", "tie"}:
        winner = "tie"

    return {
        "winner": winner,
        "confidence": float(result.get("confidence", 1)),
        "brief_reason": result.get("brief_reason", ""),
    }


def process_video(video_path, output_root):
    timings = {}
    filename, save_path = copy_video_to_media(video_path)
    output_path = Path(settings.MEDIA_ROOT) / f"geval_{Path(filename).stem}"

    log(f"Preprocessing: {video_path.name}")
    preprocessed_path = run_preprocessing(save_path, str(output_path), video_path.name, timings)

    log(f"Inference: {video_path.name}")
    result = run_inference(preprocessed_path, timings)

    log(f"Grad-CAM/ROI: {video_path.name}")
    roi_result, table_data = run_gradcam(str(output_path), video_path.name, result, timings)

    return {
        "video_path": str(video_path),
        "saved_filename": filename,
        "output_path": str(output_path),
        "result": result,
        "roi_result": roi_result,
        "table_data": table_data,
        "timings": timings,
    }


def write_csv(path, rows, fieldnames):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def append_jsonl(path, record):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as jsonl_file:
        jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_dataset_csv(path):
    with open(path, newline="", encoding="utf-8") as csv_file:
        return list(csv.DictReader(csv_file))

def group_dataset_by_video(dataset_rows):
    grouped = {}
    for row in dataset_rows:
        grouped.setdefault(row["video_name"], {})[row["setting"]] = row
    return grouped


def run_pairwise_on_dataset(args, dataset_rows):
    grouped = group_dataset_by_video(dataset_rows)
    rows = []
    video_names = sorted(grouped)
    if args.pairwise_video_limit:
        video_names = video_names[:args.pairwise_video_limit]

    total = len(video_names) * len(PAIRWISE_COMPARISONS)
    index = 0

    for video_name in video_names:
        setting_map = grouped[video_name]
        for setting_a, setting_b in PAIRWISE_COMPARISONS:
            if setting_a not in setting_map or setting_b not in setting_map:
                continue
            index += 1
            row_a = setting_map[setting_a]
            row_b = setting_map[setting_b]
            prediction = row_a["prediction"]

            log(
                f"Pairwise judging {index}/{total}: {video_name} / "
                f"{SETTING_LABELS[setting_a]} vs {SETTING_LABELS[setting_b]}"
            )
            result = judge_pairwise(
                video_name,
                prediction,
                setting_a,
                row_a["explanation"],
                setting_b,
                row_b["explanation"],
                provider=args.judge_provider,
                url=args.llm_url,
                model=args.judge_model,
                timeout=args.timeout,
            )

            winner_setting = "tie"
            if result["winner"] == "A":
                winner_setting = setting_a
            elif result["winner"] == "B":
                winner_setting = setting_b

            rows.append({
                "video_name": video_name,
                "prediction": prediction,
                "setting_a": setting_a,
                "setting_a_label": SETTING_LABELS[setting_a],
                "setting_b": setting_b,
                "setting_b_label": SETTING_LABELS[setting_b],
                "winner": result["winner"],
                "winner_setting": winner_setting,
                "winner_label": SETTING_LABELS.get(winner_setting, "tie"),
                "confidence": result["confidence"],
                "brief_reason": result["brief_reason"],
            })

    return rows


def summarize_pairwise(pairwise_rows, prediction=None):
    summary = []
    for setting_a, setting_b in PAIRWISE_COMPARISONS:
        rows = [
            row
            for row in pairwise_rows
            if row["setting_a"] == setting_a and row["setting_b"] == setting_b
            and (prediction is None or row.get("prediction") == prediction)
        ]
        if not rows:
            continue

        a_wins = sum(1 for row in rows if row["winner_setting"] == setting_a)
        b_wins = sum(1 for row in rows if row["winner_setting"] == setting_b)
        ties = sum(1 for row in rows if row["winner_setting"] == "tie")
        total = len(rows)

        summary.append({
            "prediction_group": prediction or "ALL",
            "comparison": f"{SETTING_LABELS[setting_a]} vs {SETTING_LABELS[setting_b]}",
            "setting_a": setting_a,
            "setting_b": setting_b,
            "setting_a_wins": a_wins,
            "setting_b_wins": b_wins,
            "ties": ties,
            "total": total,
            "setting_a_win_rate": round(a_wins / total, 4),
            "setting_b_win_rate": round(b_wins / total, 4),
            "tie_rate": round(ties / total, 4),
            "avg_confidence": round(sum(float(row["confidence"]) for row in rows) / total, 4),
        })

    return summary


def parse_args():
    parser = argparse.ArgumentParser(description="Collect G-Eval explanation ablation data.")
    parser.add_argument("--data-root", default=str(REPO_ROOT / "model" / "data"), help="Directory containing evaluation videos.")
    parser.add_argument("--output-dir", default=str(REPO_ROOT / "model" / "ablation_results" / "geval"), help="Output directory.")
    parser.add_argument("--sample-size", type=int, default=10)
    parser.add_argument("--balanced", action="store_true", help="Sample real/fake videos as evenly as possible based on path names.")
    parser.add_argument("--correct-only", action="store_true", help="Keep only videos whose path-derived label matches the model prediction.")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--llm-url", default=os.environ.get("DEFAKE_LLM_URL", "http://localhost:11434/api/generate"))
    parser.add_argument("--llm-provider", choices=["ollama", "openai"], default=os.environ.get("DEFAKE_LLM_PROVIDER", "ollama"))
    parser.add_argument("--llm-model", default=os.environ.get("DEFAKE_LLM_MODEL", "llama3"))
    parser.add_argument("--judge-provider", choices=["ollama", "openai"], default=os.environ.get("DEFAKE_JUDGE_PROVIDER", "ollama"))
    parser.add_argument("--judge-model", default=os.environ.get("DEFAKE_JUDGE_MODEL", os.environ.get("DEFAKE_LLM_MODEL", "llama3")))
    parser.add_argument("--timeout", type=int, default=120)
    parser.add_argument("--input-dataset", default=None, help="Existing explanation_ablation_dataset.csv path for pairwise judging.")
    parser.add_argument("--pairwise-only", action="store_true", help="Run pairwise preference judging using an existing explanation dataset CSV.")
    parser.add_argument("--pairwise-video-limit", type=int, default=None, help="Limit the number of videos for pairwise judging.")
    return parser.parse_args()


def main():
    args = parse_args()
    output_dir = Path(args.output_dir)
    dataset_jsonl = output_dir / "explanation_ablation_dataset.jsonl"
    dataset_csv = output_dir / "explanation_ablation_dataset.csv"
    pairwise_csv = output_dir / "geval_pairwise_trial.csv"
    pairwise_summary_csv = output_dir / "geval_pairwise_trial_summary.csv"
    pairwise_summary_by_prediction_csv = output_dir / "geval_pairwise_trial_summary_by_prediction.csv"

    if args.pairwise_only:
        input_dataset = Path(args.input_dataset) if args.input_dataset else dataset_csv
        log(f"Pairwise-only mode. Reading dataset: {input_dataset}")
        dataset_rows = read_dataset_csv(input_dataset)
        pairwise_rows = run_pairwise_on_dataset(args, dataset_rows)
        pairwise_summary_rows = summarize_pairwise(pairwise_rows)
        pairwise_summary_by_prediction_rows = []
        for prediction in ["FAKE", "REAL"]:
            pairwise_summary_by_prediction_rows.extend(
                summarize_pairwise(pairwise_rows, prediction=prediction)
            )

        write_csv(
            pairwise_csv,
            pairwise_rows,
            [
                "video_name",
                "prediction",
                "setting_a",
                "setting_a_label",
                "setting_b",
                "setting_b_label",
                "winner",
                "winner_setting",
                "winner_label",
                "confidence",
                "brief_reason",
            ],
        )
        write_csv(
            pairwise_summary_csv,
            pairwise_summary_rows,
            [
                "prediction_group",
                "comparison",
                "setting_a",
                "setting_b",
                "setting_a_wins",
                "setting_b_wins",
                "ties",
                "total",
                "setting_a_win_rate",
                "setting_b_win_rate",
                "tie_rate",
                "avg_confidence",
            ],
        )
        write_csv(
            pairwise_summary_by_prediction_csv,
            pairwise_summary_by_prediction_rows,
            [
                "prediction_group",
                "comparison",
                "setting_a",
                "setting_b",
                "setting_a_wins",
                "setting_b_wins",
                "ties",
                "total",
                "setting_a_win_rate",
                "setting_b_win_rate",
                "tie_rate",
                "avg_confidence",
            ],
        )

        log(f"Pairwise trial written: {pairwise_csv}")
        log(f"Pairwise trial summary written: {pairwise_summary_csv}")
        log(f"Pairwise trial summary by prediction written: {pairwise_summary_by_prediction_csv}")

        print("\n===== Pairwise Preference Trial Summary =====")
        for row in pairwise_summary_rows:
            print(row)
        print("\n===== Pairwise Preference Trial Summary by Prediction =====")
        for row in pairwise_summary_by_prediction_rows:
            print(row)
        print("=============================================\n")
        return

    if dataset_jsonl.exists():
        dataset_jsonl.unlink()

    log(f"Sampling videos from: {args.data_root}")
    if args.correct_only:
        videos = iter_candidate_videos(args.data_root, args.seed, balanced=args.balanced)
        log(f"Correct-only mode enabled. Scanning up to {len(videos)} candidate videos.")
    else:
        videos = sample_videos(args.data_root, args.sample_size, args.seed, balanced=args.balanced)
        log(f"Selected {len(videos)} videos")

    dataset_rows = []
    accepted_video_count = 0
    accepted_by_label = {"REAL": 0, "FAKE": 0}
    target_per_label = args.sample_size // 2 if args.balanced else None

    for video_index, video_path in enumerate(videos, start=1):
        if args.correct_only and accepted_video_count >= args.sample_size:
            break

        true_label = infer_label_from_path(video_path)
        if args.correct_only and args.balanced and true_label in accepted_by_label:
            if accepted_by_label[true_label] >= target_per_label:
                continue

        log(f"Video candidate {video_index}/{len(videos)}: {video_path}")
        pipeline_output = process_video(video_path, output_dir)
        result = pipeline_output["result"]
        roi_result = pipeline_output["roi_result"]

        if args.correct_only:
            prediction = result.get("Prediction")
            if true_label == "UNKNOWN":
                log(f"Skipping {video_path.name}: cannot infer ground-truth label from path")
                continue
            if prediction != true_label:
                log(f"Skipping {video_path.name}: ground_truth={true_label}, prediction={prediction}")
                continue

            accepted_video_count += 1
            if true_label in accepted_by_label:
                accepted_by_label[true_label] += 1
            log(
                f"Accepted {video_path.name}: ground_truth={true_label}, "
                f"prediction={prediction}, accepted={accepted_video_count}/{args.sample_size}, "
                f"accepted_by_label={accepted_by_label}"
            )

        for setting in SETTING_ORDER:
            evidence = build_evidence(video_path, result, roi_result, setting)
            prompt = build_explanation_prompt(evidence, setting)

            log(f"Generating explanation: {video_path.name} / {SETTING_LABELS[setting]}")
            explanation = generate_text(
                prompt,
                provider=args.llm_provider,
                url=args.llm_url,
                model=args.llm_model,
                timeout=args.timeout,
                temperature=0.0,
            )

            dataset_record = {
                "video_name": video_path.name,
                "video_path": str(video_path),
                "setting": setting,
                "setting_label": SETTING_LABELS[setting],
                "prediction": result["Prediction"],
                "probability": result["Probability"],
                "method": result["Method"],
                "evidence_json": json.dumps(evidence, ensure_ascii=False),
                "explanation": explanation,
            }
            dataset_rows.append(dataset_record)
            append_jsonl(dataset_jsonl, dataset_record)

    if args.correct_only and accepted_video_count < args.sample_size:
        log(
            f"Warning: requested {args.sample_size} correct videos but collected only "
            f"{accepted_video_count}. accepted_by_label={accepted_by_label}"
        )

    write_csv(
        dataset_csv,
        dataset_rows,
        [
            "video_name",
            "video_path",
            "setting",
            "setting_label",
            "prediction",
            "probability",
            "method",
            "evidence_json",
            "explanation",
        ],
    )
    log(f"Dataset written: {dataset_csv}")
    log(f"Dataset JSONL written: {dataset_jsonl}")


if __name__ == "__main__":
    main()
